In [1]:
"""
Aim: make figures for the manuscript Fig.1
Author: Yike Xie
Date: 3-Mar-2026 
"""

'\nAim: make figures for the manuscript Fig.1\nAuthor: Yike Xie\nDate: 3-Mar-2026 \n'

In [2]:
import sys
sys.path.append("..")
import utils

import pandas as pd
import numpy as np
import os

import scanpy as sc
import anndata as ad

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42   # TrueType fonts
rcParams['ps.fonttype'] = 42    # For EPS (optional)
rcParams['svg.fonttype'] = 'none'  # For editable text in SVG

# load data

In [4]:
metadata = pd.read_excel(
    '../../figures/manuscript_figures/tables/Table1_human_donor_information.xlsx',
    sheet_name='Pancreas', index_col=0)

tech_cols = ["snRNA-Seq", "Immunostaining", "Spatial transcriptomics", "Calcium imaging", "Slice-Seq"]
tech_mask = metadata[tech_cols].astype(str).apply(lambda s: s.str.lower().str.strip().eq("yes"))

metadata["Method"] = tech_mask.apply(
    lambda row: ", ".join([col for col, ok in row.items() if ok]),
    axis=1
)

In [3]:
# load data
adata = sc.read_h5ad('../../data/parse_snRNA_annotated_YK_raw.h5ad')
adata = adata[adata.obs['Doublets'] == 'no'].copy()

# edit the name of acinar cell subtypes refering to the paper: DOI: 10.1101/2025.10.01.679230
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype(str)

mask1 = adata.obs['cell_type'] == 'Acinar'
adata.obs.loc[mask1, 'cell_subtype'] = adata.obs.loc[mask1, 'cell_subtype2']

mask2 = adata.obs['cell_type'] == 'Endothelial'
adata.obs.loc[mask2, 'cell_subtype'] = adata.obs.loc[mask2, 'cell_subtype1']

# Fix cell_type replacement safely
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype(str).replace({
    'Signaling_acinar': 'Idling_acinar',
    'High_enzyme_acinar': 'Signaling_acinar',
})

desired_cst_order = [
    'α', 'β', 'γ', 'δ', 'Idling_acinar', 'Intermediate_acinar', 'Signaling_acinar',
    'Basal_ductal', 'Inflam_ductal_1', 'Inflam_ductal_2', 'MUC5B+_ductal',
    'Arterial_ECs', 'Venous_ECs', 'Capillary_ECs', 'Lymphatic_ECs',                   
    'Macrophages', 'Plasmablasts', 'T_cells', 'Mast_cells',
    'Activated_stellates', 'Quiescent_stellates', 'Schwann'
]
adata.obs['cell_subtype'] = adata.obs['cell_subtype'].astype('category')
adata.obs['cell_subtype'] = (
    adata.obs['cell_subtype']
    .cat.set_categories(desired_cst_order, ordered=True)
)

# normalize and logrithmize the data
adata_raw = adata.copy()
utils.normalizedata(adata, log1p=True)

AnnData object with n_obs × n_vars = 57858 × 38560
    obs: 'sample', 'doublet_score', 'Sex', 'BMI', 'T1D', 'Diabetes Duration', 'T2D', 'HbA1c (%)', 'HbA1c', 'Age', 'CIT (hours)', 'Cohort', 'RIN', 'Nuclei isolation', 'group', 'cell_type', 'cell_subtype', 'cell_subtype1', 'cell_subtype2', 'Doublets', 'batch'
    var: 'n_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'Doublets_colors', 'cst1_colors', 'log1p'
    obsm: 'X_pca', 'X_umap'

# Generate supplemenatry table 2 and 3

In [ ]:
# Table2_pseudo_bulk_cpm_by_cell_type.tsv

import scipy.sparse as sp

# Make sure we use raw counts
X = adata_raw.X
if sp.issparse(X):
    X = X.toarray()

# Add cell type info
df = pd.DataFrame(X, columns=adata.var_names)
df['cell_type'] = adata.obs['cell_type'].values

# Aggregate counts per cell type
pseudobulk = df.groupby('cell_type').sum()
# Compute library sizes per cell type
lib_sizes = pseudobulk.sum(axis=1)

# CPM normalization
cpm = pseudobulk.div(lib_sizes, axis=0) * 1e6
cpm.to_csv(
    '../../figures/manuscript_figures/tables/Table2_pseudo_bulk_cpm_by_cell_type.tsv',
    sep='\t'
    )

# Count genes with CPM > 1 per cell type
expressed_genes = (cpm > 1).sum(axis=1)

# Average across cell types
average_genes = expressed_genes.mean()
print("Genes expressed per cell type:")
print(expressed_genes)
print(f"\nOn average, {average_genes:.0f} genes were expressed (CPM > 1) per cell type.")

/tmp/ipykernel_2509757/1149622679.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk = df.groupby('cell_type').sum()


Genes expressed per cell type:
cell_type
Endocrine      18185
Acinar         17189
Ductal         17532
Endothelial    16827
Immune         17869
Stellate       18212
Schwann        13543
dtype: int64

On average, 17051 genes were expressed (CPM > 1) per cell type.


In [ ]:
# Table3_cell_type_specific_genes_rank_genes_groups.xlsx

output_filename = "Table3_cell_type_specific_genes_rank_genes_groups.xlsx"

def write_rank_genes_groups_to_excel(
    adata,
    groupby: str,
    writer: pd.ExcelWriter,
    prefix: str,
    fdr_thresh: float = 0.10,
):
    # Run DE
    sc.tl.rank_genes_groups(
        adata,
        groupby=groupby,
        method="wilcoxon",
        pts=True,
        n_genes=adata.shape[1],
    )

    groups = adata.obs[groupby].astype(str).unique().tolist()

    for g in groups:
        df = sc.get.rank_genes_groups_df(adata, group=g).copy()

        # Excel sheet name rules: <= 31 chars, unique
        base = f"{prefix}{g}"
        sheet_name = base[:31]
        # If truncation causes collisions, ExcelWriter will error; add a numeric suffix if needed
        suffix = 1
        while sheet_name in writer.sheets:
            suffix += 1
            sheet_name = (base[: (31 - len(str(suffix)) - 1)] + f"_{suffix}")[:31]
        df_sig = df[df['pvals_adj'] < 0.10].sort_values('pvals_adj')
        df_sig.to_excel(writer, sheet_name=sheet_name, index=False)

# ---- Run both analyses and write to one file ----
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    write_rank_genes_groups_to_excel(adata, groupby="cell_type", writer=writer, prefix="CT__")
    write_rank_genes_groups_to_excel(adata, groupby="cell_subtype", writer=writer, prefix="CST__")

print("Saved Excel file:", output_filename)